# Vanilla LSTM -- Helpful Amazon Review Prediction

**Binary classification**: predict whether an Amazon review is "helpful" based on the **75th percentile** of `helpful_vote`.

- Target: `helpful = 1 if helpful_vote > quantile(0.75)`, else 0
- Dataset: `amazon_reviews_s10.csv` (pre-filtered to `helpful_vote >= 1`)
- Model: single-layer unidirectional LSTM with learned embeddings

In [ ]:
import sys, subprocess

packages = [
    "pip", "setuptools", "wheel",
    "numpy", "pandas", "scikit-learn",
    "torch", "nltk",
]
print("Installing / upgrading dependencies ...")
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages]
)

import nltk
for res in ["stopwords", "wordnet", "punkt_tab"]:
    nltk.download(res, quiet=True)
print("Done.")

In [ ]:
from pathlib import Path
from collections import Counter
import re, html, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)

STOP_WORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

## 1. Load Data

In [ ]:
cwd = Path.cwd()
workspace = cwd if (cwd / "data").exists() else cwd.parent

data_path = workspace / "data" / "processed" / "amazon_reviews_s10.csv"
if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at {data_path}")

df = pd.read_csv(data_path)
df["has_image"] = df["has_image"].astype(int)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

## 2. Create Target (75th Percentile Threshold)

In [ ]:
threshold = df["helpful_vote"].quantile(0.75)
print(f"helpful_vote  min={df['helpful_vote'].min()}  "
      f"Q75={threshold}  max={df['helpful_vote'].max()}")
print(f"Threshold: helpful = 1 if helpful_vote > {threshold}")

df["helpful"] = (df["helpful_vote"] > threshold).astype(int)

print(f"\nLabel distribution:")
print(df["helpful"].value_counts())
print()
print(df["helpful"].value_counts(normalize=True))

if df["helpful"].nunique() < 2:
    raise ValueError("Only one class present -- cannot train.")

minority_frac = df["helpful"].value_counts(normalize=True).min()
print(f"\nMinority class fraction: {minority_frac:.4f}")

## 3. Text Preprocessing

In [ ]:
HTML_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
SPECIAL_RE = re.compile(r"[^a-z\s]")

def clean_text(text: str) -> str:
    text = html.unescape(str(text))
    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = text.lower()
    text = SPECIAL_RE.sub(" ", text)
    tokens = text.split()
    tokens = [LEMMATIZER.lemmatize(w) for w in tokens
              if w not in STOP_WORDS and len(w) > 1]
    return " ".join(tokens)

print("Cleaning all reviews ...")
df["clean_text"] = df["review_text"].apply(clean_text)

empty_after = (df["clean_text"].str.len() == 0).sum()
print(f"Empty reviews after cleaning: {empty_after}")
df.loc[df["clean_text"].str.len() == 0, "clean_text"] = "empty"
print(f"Final dataset size: {len(df):,}")

## 4. Train / Test Split

In [ ]:
X_text = np.array(df["clean_text"].tolist(), dtype=object)
y = np.array(df["helpful"].tolist(), dtype=np.int64)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2, shuffle=True, random_state=SEED, stratify=y,
)

print(f"Train size: {len(X_train_text):,}  |  Test size: {len(X_test_text):,}")
print(f"Train label dist: 0={int((y_train==0).sum()):,}  1={int((y_train==1).sum()):,}")
print(f"Test  label dist: 0={int((y_test==0).sum()):,}  1={int((y_test==1).sum()):,}")

## 5. Tokenization & Vocabulary (fit on train only)

In [ ]:
MAX_VOCAB = 50_000
MAX_LEN = 200
PAD_ID = 0
OOV_ID = 1

def simple_tokenize(text: str) -> list:
    return text.split()

counter = Counter()
for text in X_train_text:
    counter.update(simple_tokenize(text))

most_common = counter.most_common(MAX_VOCAB - 2)
word2idx = {word: idx + 2 for idx, (word, _) in enumerate(most_common)}
vocab_size = len(word2idx) + 2

print(f"Vocabulary size (incl. PAD+OOV): {vocab_size}")
print(f"Unique tokens in training corpus: {len(counter):,}")
print(f"Coverage by top {MAX_VOCAB - 2}: "
      f"{sum(c for _, c in most_common) / sum(counter.values()):.2%}")

def text_to_ids(text, mapping, max_len):
    ids = [mapping.get(tok, OOV_ID) for tok in simple_tokenize(text)]
    ids = ids[:max_len]
    if len(ids) < max_len:
        ids += [PAD_ID] * (max_len - len(ids))
    return ids

X_train_pad = np.array(
    [text_to_ids(t, word2idx, MAX_LEN) for t in X_train_text], dtype=np.int64
)
X_test_pad = np.array(
    [text_to_ids(t, word2idx, MAX_LEN) for t in X_test_text], dtype=np.int64
)

print(f"\nX_train_pad shape: {X_train_pad.shape}")
print(f"X_test_pad  shape: {X_test_pad.shape}")

## 6. PyTorch DataLoaders

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_pad, y_train,
    test_size=0.15, shuffle=True, random_state=SEED, stratify=y_train,
)

print(f"Training   : {X_tr.shape[0]:,}")
print(f"Validation : {X_val.shape[0]:,}")
print(f"Hold-out   : {X_test_pad.shape[0]:,}")

n_neg, n_pos = (y_tr == 0).sum(), (y_tr == 1).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(device)
print(f"\nClass counts  neg={n_neg:,}  pos={n_pos:,}")
print(f"pos_weight = {pos_weight.item():.4f}")

BATCH_SIZE = 256

def make_loader(X_pad, y, batch_size, shuffle):
    ds = TensorDataset(
        torch.tensor(X_pad, dtype=torch.long),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_tr, y_tr, BATCH_SIZE, True)
val_loader   = make_loader(X_val, y_val, BATCH_SIZE, False)
test_loader  = make_loader(X_test_pad, y_test, 512, False)

print(f"\nBatches  train={len(train_loader)}  val={len(val_loader)}  test={len(test_loader)}")

## 7. Vanilla LSTM Model

In [ ]:
class VanillaLSTM(nn.Module):
    def __init__(self, vocab_sz, emb_dim, hid_dim, pad_idx=0, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_sz, emb_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(emb_dim, hid_dim, batch_first=True)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(hid_dim, 1)

    def forward(self, text):
        x = self.emb(text)
        _, (h, _) = self.lstm(x)
        return self.fc(self.drop(h[-1])).squeeze(1)

## 8. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, epochs=5, lr=1e-3,
                pw=None, patience=3, name="Model"):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=1,
    )
    hist = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
    best_val, wait, best_state = float("inf"), 0, None

    for ep in range(1, epochs + 1):
        model.train()
        s_loss, s_ok, s_n = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            s_loss += loss.item() * xb.size(0)
            s_ok += ((torch.sigmoid(logits) >= 0.5).float() == yb).sum().item()
            s_n += xb.size(0)

        model.eval()
        v_loss, v_ok, v_n = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                v_loss += loss.item() * xb.size(0)
                v_ok += ((torch.sigmoid(logits) >= 0.5).float() == yb).sum().item()
                v_n += xb.size(0)

        tl, ta = s_loss / s_n, s_ok / s_n
        vl, va = v_loss / v_n, v_ok / v_n
        scheduler.step(vl)
        hist["loss"].append(tl); hist["val_loss"].append(vl)
        hist["accuracy"].append(ta); hist["val_accuracy"].append(va)

        print(f"  [{name}] Epoch {ep}/{epochs}  "
              f"loss={tl:.4f}  acc={ta:.4f}  val_loss={vl:.4f}  val_acc={va:.4f}")

        if vl < best_val:
            best_val, wait = vl, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                print(f"  Early stopping at epoch {ep}")
                break

    if best_state:
        model.load_state_dict(best_state)
    return hist


def evaluate_model(model, loader):
    model.eval()
    probs_list, true_list = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs_list.extend(torch.sigmoid(logits).cpu().numpy().tolist())
            true_list.extend(yb.numpy().tolist())
    yp = np.array(probs_list)
    yt = np.array(true_list, dtype=int)
    ypred = (yp >= 0.5).astype(int)
    return dict(
        accuracy=accuracy_score(yt, ypred),
        precision=precision_score(yt, ypred, zero_division=0),
        recall=recall_score(yt, ypred, zero_division=0),
        f1=f1_score(yt, ypred, zero_division=0),
        auc=roc_auc_score(yt, yp),
        y_true=yt, y_pred=ypred, y_prob=yp,
    )

## 9. Train Vanilla LSTM

In [ ]:
model = VanillaLSTM(
    vocab_sz=vocab_size, emb_dim=128, hid_dim=128,
    pad_idx=PAD_ID, drop=0.3,
).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}\n")

history = train_model(
    model, train_loader, val_loader,
    epochs=5, lr=1e-3, pw=pos_weight, name="Vanilla LSTM",
)

## 10. Evaluate on Test Set

In [ ]:
results = evaluate_model(model, test_loader)

print("Vanilla LSTM -- Test Results")
print("=" * 50)
print(f"  Accuracy  : {results['accuracy']:.4f}")
print(f"  Precision : {results['precision']:.4f}")
print(f"  Recall    : {results['recall']:.4f}")
print(f"  F1 Score  : {results['f1']:.4f}")
print(f"  ROC-AUC   : {results['auc']:.4f}")
print("=" * 50)

print("\nClassification Report:")
print(classification_report(
    results["y_true"], results["y_pred"],
    target_names=["Not Helpful", "Helpful"],
))